# 01 — Prompt-Boundary Alignment Predicts Useful Steering Layers

Reproduces paper **Fig. 2** (`gemma3_hum_concept_score_alignment.png`,
`gemma3_cs_align_coef3.0_scatter.png`), **Fig. 5**
(`gemma3_all_grid_concept_score_alignment.png`), **Fig. 6**
(`llama3_all_grid_concept_score_alignment.png`).

All inputs come from `results/statistics/` (per-layer 𝒜_c profiles) and
`results/steering_evaluations/` (the §3 grid-sweep summary CSVs).


In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from grace.analysis.load_results import load_summary_results
from grace.diagnostics.alignment import alignment_per_layer

FIG_DIR = Path('Images')
FIG_DIR.mkdir(exist_ok=True)
MODELS = {'gemma3': 'google/gemma-3-27b-it', 'llama3': 'meta-llama/Llama-3.3-70B-Instruct'}
CONCEPTS = sorted(p.stem for p in Path('concepts/gpt-5/extract').glob('*.json'))

## Fig. 2a — Alignment vs. concept score for one example concept (humorous, gemma3)


In [ ]:
concept = 'humorous'
model_name = MODELS['gemma3']
alignment = alignment_per_layer(model_name, concept, 'prompt_last')
rows = load_summary_results(model_name, concept, method='pv')
df = pd.DataFrame(rows)
df = df[df.coef == 3.0].sort_values('layer')

fig, ax1 = plt.subplots(figsize=(7, 4))
ax2 = ax1.twinx()
layers = sorted(alignment)
ax1.plot(layers, [alignment[l] for l in layers], 'b-', label='𝒜_c(ℓ) PL alignment')
ax2.plot(df.layer, df.mean_concept_score, 'r-o', label='concept score (coef=3)')
ax1.set_xlabel('Layer')
ax1.set_ylabel('Prompt-boundary alignment 𝒜_c(ℓ)', color='b')
ax2.set_ylabel('Concept score', color='r')
plt.title(f'{concept} — gemma-3-27b-it')
plt.tight_layout()
plt.savefig(FIG_DIR / 'gemma3_hum_concept_score_alignment.png', dpi=150)
plt.show()

## Fig. 2b — Pooled scatter: alignment vs. concept score across all concepts


In [ ]:
xs, ys = [], []
model_name = MODELS['gemma3']
for concept in CONCEPTS:
    try:
        a = alignment_per_layer(model_name, concept, 'prompt_last')
        rows = [r for r in load_summary_results(model_name, concept, method='pv') if r['coef'] == 3.0]
    except FileNotFoundError:
        continue
    for r in rows:
        if r['layer'] in a:
            xs.append(a[r['layer']])
            ys.append(r['mean_concept_score'])
plt.figure(figsize=(5, 5))
plt.scatter(xs, ys, alpha=0.4)
plt.xlabel('𝒜_c(ℓ) PL alignment')
plt.ylabel('Concept score (coef=3)')
plt.title('Pooled across all 20 concepts — gemma-3-27b-it')
plt.tight_layout()
plt.savefig(FIG_DIR / 'gemma3_cs_align_coef3.0_scatter.png', dpi=150)
plt.show()

## Fig. 5/6 — Per-concept alignment vs. concept-score grids (Gemma3 + Llama3)

5×4 small-multiples plot, one panel per concept. Same dual-axis layout as
Fig. 2a but rendered for every concept on each model.


In [ ]:
def grid_plot(model_name, save_path, n_rows=5, n_cols=4):
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.5, n_rows * 2.4))
    for ax, concept in zip(axes.flat, CONCEPTS):
        try:
            a = alignment_per_layer(model_name, concept, 'prompt_last')
            rows = [r for r in load_summary_results(model_name, concept, method='pv') if r['coef'] == 3.0]
        except FileNotFoundError:
            ax.set_visible(False); continue
        df = pd.DataFrame(rows).sort_values('layer')
        ax2 = ax.twinx()
        layers = sorted(a)
        ax.plot(layers, [a[l] for l in layers], 'b-')
        ax2.plot(df.layer, df.mean_concept_score, 'r-o', ms=3)
        ax.set_title(concept, fontsize=8)
        ax.tick_params(labelsize=6); ax2.tick_params(labelsize=6)
    plt.tight_layout(); plt.savefig(save_path, dpi=150); plt.show()

grid_plot(MODELS['gemma3'], FIG_DIR / 'gemma3_all_grid_concept_score_alignment.png')
grid_plot(MODELS['llama3'], FIG_DIR / 'llama3_all_grid_concept_score_alignment.png')